<a href="https://colab.research.google.com/github/prince24-web/Mechine_learning/blob/main/plant_disease_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1 — Install + imports
# ============================================================
!pip install -q kagglehub onnx onnxruntime onnxscript

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import kagglehub
import os
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.5 MB/s eta 0:00:00
Using device: cuda


In [ ]:
# ============================================================
# CELL 2 — Download PlantVillage dataset (no manual upload needed)
# ============================================================
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Dataset downloaded to:", path)

# Find the actual image folder (kaggle dataset has nested structure)
import glob
color_dir = None
for root, dirs, files in os.walk(path):
    if "color" in root.lower() and len(dirs) > 30:
        color_dir = root
        break

if not color_dir:
    # Fallback — just find the dir with the most subfolders
    candidates = [(root, len(dirs)) for root, dirs, files in os.walk(path) if len(dirs) > 30]
    color_dir = max(candidates, key=lambda x: x[1])[0]

print("Using image directory:", color_dir)
print("Number of classes found:", len(os.listdir(color_dir)))

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Dataset downloaded to: /kaggle/input/plantvillage-dataset
Using image directory: /kaggle/input/plantvillage-dataset/plantvillage dataset/color
Number of classes found: 38


In [ ]:
# ============================================================
# CELL 3 — Data loaders with augmentation
# ============================================================
IMG_SIZE = 224
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(color_dir, transform=train_transform)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Found {NUM_CLASSES} classes")
print(f"Total images: {len(full_dataset)}")

# Save class names now — you'll need this list later in vision.py
with open("class_names.txt", "w") as f:
    for name in CLASS_NAMES:
        f.write(name + "\n")

# 85/15 train/val split
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
val_ds.dataset.transform = val_transform  # val set shouldn't use augmentation

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Found 38 classes
Total images: 54305
Train: 46159 | Val: 8146


In [ ]:
# ============================================================
# CELL 4 — Build model: pretrained MobileNetV3 + transfer learning
# ============================================================
model = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")

# Freeze the feature extractor — we only train the classifier head
for param in model.features.parameters():
    param.requires_grad = False

# Replace final classifier layer for our number of classes
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

# Only optimize the new classifier layer — fast training
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print("Model ready. Trainable params:")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  {trainable:,} / {total:,}")

Model ready. Trainable params:
  1,278,758 / 4,250,710


In [ ]:
# ============================================================
# CELL 5 — Train (5-8 epochs, ~3-4 min per epoch on T4)
# ============================================================
EPOCHS = 6

for epoch in range(EPOCHS):
    start = time.time()

    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    elapsed = time.time() - start

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss/train_total:.4f} | "
          f"Train Acc: {100*train_correct/train_total:.1f}% | "
          f"Val Acc: {100*val_correct/val_total:.1f}% | "
          f"{elapsed:.0f}s")

print("\n✓ Training complete")

Epoch 1/6 | Train Loss: 0.3055 | Train Acc: 91.2% | Val Acc: 96.2% | 233s
Epoch 2/6 | Train Loss: 0.1070 | Train Acc: 96.5% | Val Acc: 96.5% | 161s
Epoch 3/6 | Train Loss: 0.0701 | Train Acc: 97.6% | Val Acc: 96.3% | 154s
Epoch 4/6 | Train Loss: 0.0321 | Train Acc: 98.9% | Val Acc: 97.5% | 154s
Epoch 5/6 | Train Loss: 0.0238 | Train Acc: 99.2% | Val Acc: 97.7% | 155s
Epoch 6/6 | Train Loss: 0.0220 | Train Acc: 99.3% | Val Acc: 97.8% | 160s

✓ Training complete


In [ ]:
# ============================================================
# CELL 6 — Export to ONNX
# ============================================================
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

onnx_path = "plant_disease.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print(f"✓ Exported to {onnx_path}")

# Verify it loads correctly
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model verified — valid format")

size_mb = os.path.getsize(onnx_path) / (1024*1024)
print(f"✓ Model size: {size_mb:.1f} MB")

/tmp/ipykernel_796/477955430.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0620 13:52:04.179000 796 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: adapt: Assertion `node-

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✓ Exported to plant_disease.onnx
✓ ONNX model verified — valid format
✓ Model size: 0.3 MB


In [ ]:
# ============================================================
# CELL 7 — Quick sanity test with onnxruntime (CPU, matches your laptop)
# ============================================================
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# Grab one real validation image and test it
test_img, test_label = val_ds[0]
test_input = test_img.unsqueeze(0).numpy()

outputs = session.run(None, {"input": test_input})
pred_class = np.argmax(outputs[0])

print(f"True label: {CLASS_NAMES[test_label]}")
print(f"Predicted : {CLASS_NAMES[pred_class]}")
print(f"Match: {'✓ YES' if pred_class == test_label else '✗ no — check training'}")

True label: Orange___Haunglongbing_(Citrus_greening)
Predicted : Orange___Haunglongbing_(Citrus_greening)
Match: ✓ YES


In [ ]:
# ============================================================
# CELL 8 — Download both files to your laptop
# ============================================================
from google.colab import files

files.download("plant_disease.onnx")
files.download("class_names.txt")

print("\n✓ Both files downloading now.")
print("Move plant_disease.onnx to: agrisense/data/models/")
print("Use class_names.txt to verify your CLASS list order matches vision.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Both files downloading now.
Move plant_disease.onnx to: agrisense/data/models/
Use class_names.txt to verify your CLASS list order matches vision.py


In [ ]:
# ============================================================
# CELL 9 — Check for and download the external data file
# ============================================================
import os

print("Files in current directory:")
for f in os.listdir("."):
    if "plant_disease" in f:
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {f} — {size_mb:.2f} MB")

Files in current directory:
  plant_disease.onnx.data — 16.19 MB
  plant_disease.onnx — 0.33 MB


In [ ]:
from google.colab import files
files.download("plant_disease.onnx.data")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import shutil
import os # Added import os

# Create a directory to save sample images
sample_dir = "sample_images"
if os.path.exists(sample_dir):
    shutil.rmtree(sample_dir)
os.makedirs(sample_dir)

def save_and_display_samples(dataset, num_samples, prefix, class_names, transform_original):
    count = 0
    # Undo the normalization and ToTensor for saving/displaying
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )

    for i in range(len(dataset)):
        if count >= num_samples:
            break

        img_tensor, label_idx = dataset[i]

        # If the transform was applied to the entire dataset (which is the case for Subset objects),
        # we need to get the original image first, then apply the transform that does NOT normalize.
        # This is a bit tricky with random_split, as the transform is set for the full_dataset.
        # A simpler way is to reverse the normalization on the tensor.

        # Make a copy to avoid modifying the original tensor if it's reused
        img_to_save = img_tensor.clone().cpu()
        img_to_save = inv_normalize(img_to_save)
        img_to_save = transforms.ToPILImage()(img_to_save)

        # Save the image
        filename = os.path.join(sample_dir, f"{prefix}_{count}_{class_names[label_idx]}.png")
        img_to_save.save(filename)
        count += 1
    print(f"Saved {count} {prefix} images to {sample_dir}/")

# Get the original transform for validation set (which includes Resize and ToTensor before normalization)
# We need the original val_transform without augmentation for display, but to reverse normalization
# we need the inverse of the normalize transform used.

# For simplicity, we will reverse the normalization on the tensor and convert to PIL image.

print("Saving training samples...")
save_and_display_samples(train_ds, 20, "train_sample", CLASS_NAMES, train_transform)

print("Saving validation samples...")
save_and_display_samples(val_ds, 20, "val_sample", CLASS_NAMES, val_transform)

# Zip the directory
shutil.make_archive("plant_disease_samples", "zip", sample_dir)
print("✓ Zipped sample images into plant_disease_samples.zip")

Saving training samples...
Saved 20 train_sample images to sample_images/
Saving validation samples...
Saved 20 val_sample images to sample_images/
✓ Zipped sample images into plant_disease_samples.zip


In [ ]:
from google.colab import files

files.download("plant_disease_samples.zip")
print("✓ Downloading plant_disease_samples.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloading plant_disease_samples.zip
